# P7 EDA — 장기 horizon 오차의 원인 특정

목적은 하나다. **36h~72h horizon의 오차를 줄이려면 무엇을 고쳐야 하는가**를 데이터로 확정한다.

확인할 것:

1. 시간축 복원 — train 이 월 기반 분할(1~8월 블록 × 11년)인지
2. horizon 별 persistence 기준선 — 어느 horizon 부터 코로나홀이 유일한 신호원인지
3. 원반 반지름의 계절 변동 — 지구-태양 거리 ±1.7% 가 train/test 계통 편향이 되는지
4. **탄도 정렬 실증** — 코로나홀 면적과 타깃의 상관이 최대가 되는 지연이 실제로 몇 시간인지
5. 채널(193/211/AND) · 임계 · 위도 밴드별 신호 세기 비교
6. 코로나홀 면적 시계열의 이상치 빈도

모델 학습은 하지 않는다. 데이터만 본다.

## 0. 설정

In [ ]:
from pathlib import Path
import json, math, os, time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

DATA_ROOT_CANDIDATES = [
    Path(os.getenv("SW_DATA_ROOT", "")) if os.getenv("SW_DATA_ROOT") else None,
    Path("public_dataset/competition_dataset_6h"),
    Path("/home/jovyan/public_dataset/competition_dataset_6h"),
    Path("public/public_dataset/competition_dataset_6h"),
    Path("/home/jovyan/public/public_dataset/competition_dataset_6h"),
    Path("dataset"), Path("/home/jovyan/dataset"),
]
DATA_ROOT = None
for candidate in DATA_ROOT_CANDIDATES:
    if candidate is not None and (candidate / "train/inputs.csv").exists():
        DATA_ROOT = candidate
        break
if DATA_ROOT is None:
    raise FileNotFoundError("데이터 경로 없음: "
                            + ", ".join(str(c) for c in DATA_ROOT_CANDIDATES if c))

WORK_DIR = Path("work")
CACHE_ROOT = WORK_DIR / "cache"
EDA_DIR = WORK_DIR / "eda"
for directory in (CACHE_ROOT, EDA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

IMAGE_COLUMNS = [f"image_{i:02d}" for i in range(20)]
WIND_COLUMNS = [f"wind_{i:02d}" for i in range(20)]
TARGET_COLUMNS = [f"target_{i:02d}" for i in range(12)]
HORIZONS = np.arange(1, 13) * 6
CHANNELS = ("193", "211")
AU_KM = 1.496e8

train_inputs = pd.read_csv(DATA_ROOT / "train/inputs.csv")
train_targets_frame = pd.read_csv(DATA_ROOT / "train/targets.csv")
val_inputs = pd.read_csv(DATA_ROOT / "validation/inputs.csv")
val_targets_frame = pd.read_csv(DATA_ROOT / "validation/targets.csv")
test_inputs = pd.read_csv(DATA_ROOT / "test/inputs.csv")

assert train_inputs.sample_id.tolist() == train_targets_frame.sample_id.tolist()
assert val_inputs.sample_id.tolist() == val_targets_frame.sample_id.tolist()

train_wind = train_inputs[WIND_COLUMNS].to_numpy(np.float64)
val_wind = val_inputs[WIND_COLUMNS].to_numpy(np.float64)
train_targets = train_targets_frame[TARGET_COLUMNS].to_numpy(np.float64)
val_targets = val_targets_frame[TARGET_COLUMNS].to_numpy(np.float64)

print("data:", DATA_ROOT.resolve())
print(f"train {len(train_inputs):,} / validation {len(val_inputs):,} / test {len(test_inputs):,}")
print(f"train target mean {train_targets.mean():.1f}  std {train_targets.std():.1f}")
print(f"train wind   결측 {np.isnan(train_wind).sum()}   val wind 결측 {np.isnan(val_wind).sum()}")

## 1. 시간축 복원 — 파일명 사슬

`inputs.csv` 에는 타임스탬프가 없다. **행 순서가 시간 순서라는 보장도 없다** (실제로 아니었다).

행 순서에 의존하지 않는 방법을 쓴다. 한 행의 `image_00..image_19` 는 연속된 20 시점이므로
**인접 쌍이 곧 "직후" 관계**다. 모든 행에서 이 관계를 모아 사슬로 이으면 전체 프레임의 시간축이
복원된다. 사슬이 끊기는 지점이 실제 데이터 공백이자 자연스러운 블록 경계다.

복원이 옳다면 **시간 순으로 재정렬한 뒤에는 `wind` 체인도 맞아떨어져야 한다** — 교차 검증으로 함께 확인한다.

**Son et al. 2023 과 같은 월 기반 분할이라면 (train = 1~8월) 블록이 연도마다 하나씩, 총 11개 나와야 한다.**

In [ ]:
def reconstruct_frame_chains(inputs):
    """이미지 파일명만으로 프레임의 시간축을 복원한다. 행 순서에 의존하지 않는다.

    한 행의 image_00..image_19 는 연속된 20 시점이므로 인접 쌍이 곧 '직후' 관계다.
    모든 행에서 이 관계를 모아 사슬로 이으면 전체 프레임의 시간 순서가 나온다.
    """
    images = inputs[IMAGE_COLUMNS].to_numpy()
    successor, predecessor, conflicts = {}, {}, 0
    for row in images:
        for current, following in zip(row[:-1], row[1:]):
            if successor.setdefault(current, following) != following:
                conflicts += 1
            if predecessor.setdefault(following, current) != current:
                conflicts += 1
    names = set(images.ravel().tolist())
    chains, visited = [], set()
    for head in sorted(names - set(predecessor)):
        chain, node = [], head
        while node is not None and node not in visited:
            visited.add(node)
            chain.append(node)
            node = successor.get(node)
        chains.append(chain)
    return chains, conflicts, sorted(names - visited)


def sample_time_blocks(inputs, chains):
    """샘플을 시간 순으로 정렬하고 연속 구간(행 인덱스 배열)으로 나눈다."""
    position = {name: (c, o)
                for c, chain in enumerate(chains) for o, name in enumerate(chain)}
    first = inputs[IMAGE_COLUMNS[0]].to_numpy()
    last = inputs[IMAGE_COLUMNS[-1]].to_numpy()
    chain_id = np.array([position[n][0] for n in first])
    offset = np.array([position[n][1] for n in first])
    span_ok = np.array([position[b][0] == position[a][0]
                        and position[b][1] - position[a][1] == 19
                        for a, b in zip(first, last)])
    order = np.lexsort((offset, chain_id))
    blocks, run = [], [order[0]]
    for previous, current in zip(order[:-1], order[1:]):
        if chain_id[current] == chain_id[previous] and offset[current] == offset[previous] + 1:
            run.append(current)
        else:
            blocks.append(np.array(run))
            run = [current]
    blocks.append(np.array(run))
    return blocks, order, span_ok


FRAME_CHAINS, SAMPLE_BLOCKS = {}, {}
for name, inputs in [("train", train_inputs), ("validation", val_inputs), ("test", test_inputs)]:
    chains, conflicts, leftover = reconstruct_frame_chains(inputs)
    blocks, order, span_ok = sample_time_blocks(inputs, chains)
    lengths = np.array([len(b) for b in blocks])

    wind = inputs[WIND_COLUMNS].to_numpy(np.float64)[order]
    chain_wind = np.all(np.isclose(wind[:-1, 1:], wind[1:, :-1], equal_nan=True), axis=1)

    print(f"\n=== {name} ===")
    print(f"  프레임 사슬 {len(chains)}개 | 충돌 {conflicts} | 미방문 {len(leftover)}")
    print(f"  사슬 길이 상위 12: {sorted((len(c) for c in chains), reverse=True)[:12]}")
    print(f"  윈도우가 사슬 안에서 20연속인 비율 : {span_ok.mean():.3f}")
    print(f"  행 순서 == 시간 순서 : {np.array_equal(order, np.arange(len(inputs)))}")
    print(f"  시간 정렬 후 wind 체인 일치율 : {chain_wind.mean():.3f}")
    print(f"  샘플 블록 {len(blocks)}개 | 최대 {lengths.max()} | 50 이상 {int((lengths >= 50).sum())}개")
    big = lengths[lengths >= 50]
    if len(big):
        span_days = (big - 1) * 0.25 + 5.0          # 샘플 간 6h + 윈도우 5일
        print(f"  50샘플 이상 블록 평균 {span_days.mean():.0f}일 "
              f"({span_days.mean() / 30.4:.1f}개월)")
    print(f"  블록 길이 상위 15: {np.sort(lengths)[::-1][:15].tolist()}")
    FRAME_CHAINS[name] = chains
    SAMPLE_BLOCKS[name] = blocks

TRAIN_BLOCKS = [b for b in SAMPLE_BLOCKS["train"] if len(b) >= 50]
print(f"\n>>> CV 폴드 후보 블록: {len(TRAIN_BLOCKS)}개")
print(">>> 11개 내외 · 각 7~8개월이면 월 기반 분할(1~8월 × 11년) 확정")
print(">>> '행 순서 == 시간 순서' 가 False 면 행이 섞여 있다는 뜻이고,")
print("    P6 의 validation 2-fold 시간 분할은 근거가 없었다는 뜻이다")

## 2. horizon 별 persistence 기준선

`target_h ≈ wind_19` (마지막 관측값을 그대로 유지) 가 얼마나 버티는지를 horizon 별로 본다.
**persistence 가 무너지는 지점부터가 코로나홀이 유일한 신호원인 구간**이고, 그곳이 우리 표적이다.

In [ ]:
def official_rmse(y_true, y_pred):
    per_horizon = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
    return float(per_horizon.mean()), per_horizon


def horizon_table(wind, targets, label):
    persistence = np.repeat(wind[:, -1:], 12, axis=1)
    _, persistence_rmse = official_rmse(targets, persistence)
    climatology = np.full_like(targets, targets.mean())
    _, climatology_rmse = official_rmse(targets, climatology)
    correlation = np.array([np.corrcoef(wind[:, -1], targets[:, h])[0, 1] for h in range(12)])
    frame = pd.DataFrame({
        "horizon": HORIZONS,
        "target_std": targets.std(axis=0),
        "persistence_rmse": persistence_rmse,
        "climatology_rmse": climatology_rmse,
        "persist_corr": correlation,
    })
    frame["persist_gain"] = frame.climatology_rmse - frame.persistence_rmse
    print(f"\n=== {label} ===")
    print(frame.round(2).to_string(index=False))
    return frame


train_baseline = horizon_table(train_wind, train_targets, "train")
val_baseline = horizon_table(val_wind, val_targets, "validation")

# P3 결과가 남아 있으면 함께 본다
p3_metrics = None
for candidate in [Path("work/outputs_p3/validation_metrics.csv"),
                  Path("work/outputs/validation_metrics.csv")]:
    if candidate.exists():
        p3_metrics = pd.read_csv(candidate)
        print(f"\n=== 기존 모델 검증 지표 ({candidate}) ===")
        print(p3_metrics.round(2).to_string(index=False))
        break
if p3_metrics is None:
    print("\n[주의] 기존 validation_metrics.csv 없음 — P3 노트북을 한 번 돌려 horizon별 RMSE를 확보할 것")

figure, axis = plt.subplots(figsize=(8, 4.5))
axis.plot(HORIZONS, val_baseline.persistence_rmse, "o-", label="persistence (val)")
axis.plot(HORIZONS, val_baseline.climatology_rmse, "s--", label="climatology (val)")
if p3_metrics is not None:
    column = next((c for c in p3_metrics.columns if "rmse" in c.lower()
                   and "persist" not in c.lower()), None)
    if column is not None:
        axis.plot(HORIZONS, p3_metrics[column], "^-", label=f"model ({column})")
axis.set_xlabel("horizon [hour]"); axis.set_ylabel("RMSE [km/s]")
axis.grid(alpha=0.3); axis.legend(); axis.set_title("baseline RMSE by horizon")
plt.tight_layout(); plt.savefig(EDA_DIR / "horizon_baseline.png", dpi=140); plt.show()

## 3. 원반 검출 · 계절 변동 점검

지구-태양 거리는 근일점(1월)에서 원일점(7월)까지 **약 ±1.7%** 변한다.
겉보기 태양 반지름이 그만큼 변하면 원반 면적이 **±3.4%** 흔들린다.
train 은 1~8월, test 는 10~12월이므로 **계절과 결부된 계통 편향**이 될 수 있다.

P3 는 원반을 **평균 영상에서 한 번만** 검출해 전 구간에 같은 마스크를 쓴다.
프레임별로 반지름을 재서 실제로 흔들리는지 확인한다.

In [ ]:
def unique_filenames(inputs):
    return sorted(pd.unique(inputs[IMAGE_COLUMNS].to_numpy().ravel()).tolist())


train_files = unique_filenames(train_inputs)
print(f"train 고유 이미지 {len(train_files):,}장")


def load_pair(split, name):
    planes = []
    for channel in CHANNELS:
        with Image.open(DATA_ROOT / split / channel / name) as image:
            planes.append(np.asarray(image.convert("L"), dtype=np.float32))
    return np.stack(planes)


def measure_disk(frame, threshold_ratio=0.15):
    """단일 프레임에서 원반 중심과 반지름을 잰다."""
    mean_plane = frame.mean(axis=0)
    mask = mean_plane > mean_plane.max() * threshold_ratio
    if mask.sum() < 100:
        return np.nan, np.nan, np.nan, np.nan
    ys, xs = np.nonzero(mask)
    radius = math.sqrt(mask.sum() / math.pi)
    return float(ys.mean()), float(xs.mean()), float(radius), float(mean_plane[mask].mean())


GEOMETRY_SAMPLES = 800
geometry_path = EDA_DIR / "disk_geometry.csv"
if geometry_path.exists():
    geometry = pd.read_csv(geometry_path)
    print(f"기존 결과 재사용: {geometry_path}")
else:
    indexes = np.unique(np.linspace(0, len(train_files) - 1, GEOMETRY_SAMPLES).astype(int))
    records, started = [], time.perf_counter()
    for order, index in enumerate(indexes):
        frame = load_pair("train", train_files[index])
        center_y, center_x, radius, brightness = measure_disk(frame)
        records.append({"file_index": int(index), "center_y": center_y, "center_x": center_x,
                        "radius": radius, "disk_brightness": brightness,
                        "mean_193": float(frame[0].mean()), "mean_211": float(frame[1].mean())})
        if (order + 1) % 200 == 0:
            print(f"  {order + 1}/{len(indexes)} ({time.perf_counter() - started:.0f}s)", flush=True)
    geometry = pd.DataFrame(records)
    geometry.to_csv(geometry_path, index=False)
    print(f"저장: {geometry_path} ({time.perf_counter() - started:.0f}s)")

radius = geometry.radius.to_numpy()
print(f"\n반지름 평균 {np.nanmean(radius):.2f}px  표준편차 {np.nanstd(radius):.2f}px "
      f"({np.nanstd(radius)/np.nanmean(radius):.2%})")
print(f"반지름 최소 {np.nanmin(radius):.2f}  최대 {np.nanmax(radius):.2f}  "
      f"진폭 {(np.nanmax(radius)-np.nanmin(radius))/np.nanmean(radius):.2%}")
print("\n>>> 진폭이 3% 내외면 지구-태양 거리 변동이 그대로 들어온 것 → 프레임별 원반 정규화 필요")
print(">>> 진폭이 0.5% 미만이면 주최측이 이미 정규화한 것 → 고정 마스크 유지 가능")

figure, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(geometry.file_index, geometry.radius, ".", ms=3)
axes[0].set_title("disk radius vs time index"); axes[0].set_ylabel("radius [px]")
axes[1].plot(geometry.file_index, geometry.center_y, ".", ms=3, label="center y")
axes[1].plot(geometry.file_index, geometry.center_x, ".", ms=3, label="center x")
axes[1].legend(); axes[1].set_title("disk center vs time index")
axes[2].plot(geometry.file_index, geometry.mean_193, ".", ms=3, label="193")
axes[2].plot(geometry.file_index, geometry.mean_211, ".", ms=3, label="211")
axes[2].legend(); axes[2].set_title("frame mean intensity (instrument degradation)")
for axis in axes:
    axis.set_xlabel("unique image index"); axis.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(EDA_DIR / "disk_geometry.png", dpi=140); plt.show()

## 4. 코로나홀 면적 시계열 확보

탄도 정렬을 실증하려면 **연속 구간**의 코로나홀 면적 시계열이 필요하다.
전체 10,139장을 다 읽으면 30분이 넘으므로, **가장 긴 연속 블록 2개**에 해당하는 이미지만 읽는다.

한 번 읽는 김에 뒤에서 쓸 것을 전부 뽑아 둔다.

- 격자: 위도 6 × 경도 6 (뒤에서 3×5, 4×3, 6×3 등으로 합산 파생 가능)
- 채널 모드: `193 단독` / `211 단독` / `193 AND 211`
- 임계: 정규화값(픽셀 / 원반 중앙값) 기준 여러 단계 + **밝은 쪽 1단계**(활성영역)
- 비교는 `<=` (HANDOFF §7.1)

In [ ]:
EDA_GRID = (6, 6)                     # (위도, 경도)
DARK_CUTS = (0.30, 0.45, 0.60, 0.75)  # 정규화값이 이 이하이면 어두움
BRIGHT_CUT = 1.60                     # 정규화값이 이 이상이면 활성영역
DISK_MARGIN = 0.95
N_LAT, N_LON = EDA_GRID
N_CELLS = N_LAT * N_LON
MODES = ("193", "211", "AND")

# 가장 긴 연속 블록 2개. 블록이 짧으면 개수를 늘려 표본을 확보한다.
candidates = TRAIN_BLOCKS if TRAIN_BLOCKS else SAMPLE_BLOCKS["train"]
ranked = sorted(candidates, key=len, reverse=True)
selected, total = [], 0
for block in ranked:
    selected.append(block)
    total += len(block)
    if total >= 2400 and len(selected) >= 2:
        break
sample_rows = np.concatenate(selected)
print(f"선택 블록 {len(selected)}개, 길이 {[len(b) for b in selected][:10]} "
      f"-> 샘플 {len(sample_rows):,}개")

# 파일 목록은 반드시 **시간 순**으로 둔다. 파일명이 익명화돼 있으면 알파벳 순은 시간 순이 아니고,
# 뒤의 이상치 시계열·상관 분석이 전부 의미를 잃는다.
needed = set(train_inputs.iloc[sample_rows][IMAGE_COLUMNS].to_numpy().ravel().tolist())
block_files = [name for chain in FRAME_CHAINS["train"] for name in chain if name in needed]
assert len(block_files) == len(needed), "사슬에 없는 파일이 있다"
file_position = {name: i for i, name in enumerate(block_files)}
print(f"읽을 고유 이미지 {len(block_files):,}장 (전체의 {len(block_files)/len(train_files):.0%})")


def build_geometry(reference_frame):
    side = reference_frame.shape[-1]
    center_y, center_x, radius, _ = measure_disk(reference_frame)
    effective = radius * DISK_MARGIN
    grid_y, grid_x = np.mgrid[0:side, 0:side].astype(np.float64)
    disk = np.sqrt((grid_y - center_y) ** 2 + (grid_x - center_x) ** 2) <= effective
    lat = np.clip((grid_y - (center_y - effective)) / (2 * effective) * N_LAT, 0, N_LAT - 1e-6)
    lon = np.clip((grid_x - (center_x - effective)) / (2 * effective) * N_LON, 0, N_LON - 1e-6)
    cell = (lat.astype(np.int64) * N_LON + lon.astype(np.int64))[disk]
    counts = np.maximum(np.bincount(cell, minlength=N_CELLS), 1).astype(np.float32)
    return disk, cell, counts, (center_y, center_x, effective)


CH_PATH = EDA_DIR / f"ch_block_{N_LAT}x{N_LON}.npy"
if CH_PATH.exists():
    ch_area = np.load(CH_PATH)
    print(f"기존 결과 재사용: {CH_PATH}  shape={ch_area.shape}")
else:
    reference = load_pair("train", block_files[len(block_files) // 2])
    disk_mask, cell_id, cell_counts, disk_info = build_geometry(reference)
    print(f"원반: center=({disk_info[0]:.1f},{disk_info[1]:.1f}) r={disk_info[2]:.1f} "
          f"픽셀 {int(disk_mask.sum()):,}")

    n_levels = len(DARK_CUTS) + 1              # 어두운 단계들 + 밝은 단계 1개
    ch_area = np.zeros((len(block_files), len(MODES), n_levels, N_CELLS), np.float32)
    started = time.perf_counter()
    for index, name in enumerate(block_files):
        on_disk = load_pair("train", name)[:, disk_mask]
        median = np.median(on_disk[:, ::4], axis=1, keepdims=True)
        normalized = on_disk / np.maximum(median, 1e-3)
        dark_193 = [normalized[0] <= cut for cut in DARK_CUTS]
        dark_211 = [normalized[1] <= cut for cut in DARK_CUTS]
        for level in range(len(DARK_CUTS)):
            for mode_index, selection in enumerate(
                    (dark_193[level], dark_211[level],
                     np.logical_and(dark_193[level], dark_211[level]))):
                ch_area[index, mode_index, level] = (
                    np.bincount(cell_id[selection], minlength=N_CELLS) / cell_counts)
        bright_193 = normalized[0] >= BRIGHT_CUT
        bright_211 = normalized[1] >= BRIGHT_CUT
        for mode_index, selection in enumerate(
                (bright_193, bright_211, np.logical_and(bright_193, bright_211))):
            ch_area[index, mode_index, -1] = (
                np.bincount(cell_id[selection], minlength=N_CELLS) / cell_counts)
        if (index + 1) % 500 == 0 or index + 1 == len(block_files):
            print(f"  {index + 1}/{len(block_files)} "
                  f"({time.perf_counter() - started:.0f}s)", flush=True)
    np.save(CH_PATH, ch_area)
    print(f"저장: {CH_PATH}  shape={ch_area.shape}")

LEVEL_NAMES = [f"dark<={c}" for c in DARK_CUTS] + [f"bright>={BRIGHT_CUT}"]
print("\n면적 비율 (원반 전체 평균):")
for mode_index, mode in enumerate(MODES):
    values = ch_area[:, mode_index].mean(axis=(0, 2))
    print(f"  {mode:4s} " + "  ".join(
        f"{LEVEL_NAMES[l]}={ch_area[:, mode_index, l].mean():.4f}"
        for l in range(ch_area.shape[2])))

## 5. 탄도 정렬 실증 — **이번 EDA의 핵심**

윈도우 인덱스 `j`(0~19, 19가 마지막 관측 T0)의 프레임은 시각 `T0 − 6(19−j)` 이고,
horizon `h`(1~12)의 타깃은 시각 `T0 + 6h` 다. 따라서 둘 사이의 지연은

$$\mathrm{lag}(j,h) = 6\,(h + 19 - j)\ \text{시간}$$

이고 6시간부터 186시간까지 6시간 간격으로 나온다.

태양풍 전달 시간이 τ = 1AU/v 라면 **상관은 lag ≈ τ 부근에서 최대**여야 하고,
그 값은 **horizon 과 무관**해야 한다 (전달 시간은 예측 리드와 관계없다).

- 봉우리가 τ(v=500)=83h 근처면 현재 `TRANSIT_SPEEDS` 가 타당하다
- 봉우리가 크게 어긋나면 **탄도 정렬 공식부터 틀린 것**이고, 그게 장기 오차의 원인이다

In [ ]:
CENTRAL_LON = N_LON // 2
EQUATOR_LAT = N_LAT // 2
AND_MODE = MODES.index("AND")
REFERENCE_LEVEL = DARK_CUTS.index(0.45) if 0.45 in DARK_CUTS else 1

index_matrix = np.asarray(
    [[file_position[n] for n in row]
     for row in train_inputs.iloc[sample_rows][IMAGE_COLUMNS].itertuples(index=False, name=None)],
    dtype=np.int64)
block_targets = train_targets[sample_rows]
block_wind = train_wind[sample_rows]
# 모델이 실제로 학습하는 양은 잔차다. persistence 를 통한 간접 상관을 걷어내고 본다.
block_residual = block_targets - block_wind[:, -1:]
print(f"샘플 {index_matrix.shape[0]:,} × 윈도우 {index_matrix.shape[1]}")


def central_series(mode_index, level, lat_band=None):
    """(n_samples, 20) 중앙 자오선 코로나홀 면적 시계열."""
    if lat_band is None:
        lat_rows = [EQUATOR_LAT]
        if N_LAT % 2 == 0:
            lat_rows = [N_LAT // 2 - 1, N_LAT // 2]
    else:
        lat_rows = [lat_band]
    cells = [row * N_LON + CENTRAL_LON for row in lat_rows]
    if N_LON % 2 == 0:
        cells += [row * N_LON + (N_LON // 2 - 1) for row in lat_rows]
    area = ch_area[:, mode_index, level][:, cells].mean(axis=1)   # (n_files,)
    return area[index_matrix]                                     # (n_samples, 20)


MIN_LAG_HOURS = 24          # 이보다 짧은 지연은 물리적으로 태양풍 전달이 불가능하다


def lag_correlation(series, targets):
    """lag(시간) -> 평균 상관 / 표본 쌍 수 / (horizon, window) 상관 행렬."""
    accumulator, table = {}, np.full((12, 20), np.nan)
    for h in range(12):
        target = targets[:, h]
        for j in range(20):
            column = series[:, j]
            if column.std() < 1e-12 or target.std() < 1e-12:
                continue
            correlation = float(np.corrcoef(column, target)[0, 1])
            table[h, j] = correlation
            accumulator.setdefault(6 * (h + 1 + 19 - j), []).append(correlation)
    lags = np.array(sorted(accumulator))
    values = np.array([np.mean(accumulator[l]) for l in lags])
    counts = np.array([len(accumulator[l]) for l in lags])
    return lags, values, counts, table


def peak_of(lags, values):
    valid = lags >= MIN_LAG_HOURS
    if not valid.any() or np.all(np.isnan(values[valid])):
        return np.nan, np.nan
    index = np.nanargmax(np.abs(values[valid]))
    return float(lags[valid][index]), float(values[valid][index])


def best_lag_per_horizon(table):
    result = []
    for h in range(12):
        row = table[h].copy()
        for j in range(20):
            if 6 * (h + 1 + 19 - j) < MIN_LAG_HOURS:
                row[j] = np.nan
        result.append(np.nan if np.all(np.isnan(row))
                      else 6 * (h + 1 + 19 - int(np.nanargmax(row))))
    return np.array(result, dtype=float)


series = central_series(AND_MODE, REFERENCE_LEVEL)
lags, curve_target, pair_counts, table_target = lag_correlation(series, block_targets)
_, curve_residual, _, table_residual = lag_correlation(series, block_residual)

peak_lag, peak_corr = peak_of(lags, curve_target)
peak_lag_res, peak_corr_res = peak_of(lags, curve_residual)
implied_speed = AU_KM / (peak_lag * 3600.0)
implied_speed_res = AU_KM / (peak_lag_res * 3600.0)

print(f"\n[타깃 기준]  최대 지연 {peak_lag:.0f} h ({peak_lag/24:.2f} 일) "
      f"-> 함의 속도 {implied_speed:.0f} km/s,  상관 {peak_corr:.3f}")
print(f"[잔차 기준]  최대 지연 {peak_lag_res:.0f} h ({peak_lag_res/24:.2f} 일) "
      f"-> 함의 속도 {implied_speed_res:.0f} km/s,  상관 {peak_corr_res:.3f}")
print("\n잔차 = target - wind_19. 모델이 실제로 학습하는 양이고, persistence 로 설명되는 부분을")
print("걷어낸 것이다. **장기 horizon 개선 여지는 잔차 상관이 결정한다.**")
print("\n참고 — 전달 시간")
for speed in (300.0, 350.0, 500.0, 700.0, 800.0):
    print(f"  v={speed:5.0f} km/s -> tau = {AU_KM/speed/3600:6.1f} h")

best_target = best_lag_per_horizon(table_target)
best_residual = best_lag_per_horizon(table_residual)
horizon_frame = pd.DataFrame({
    "horizon": HORIZONS,
    "best_lag_target": best_target,
    "best_lag_residual": best_residual,
    "implied_v_residual": AU_KM / (best_residual * 3600.0),
    "max_corr_residual": np.nanmax(np.abs(table_residual), axis=1),
})
print("\nhorizon별 최적 지연")
print(horizon_frame.round(1).to_string(index=False))
print(f"\nbest_lag_residual 산포 (표준편차) : {np.nanstd(best_residual):.1f} h")
print(">>> horizon 에 무관하게 일정하면 탄도 정렬 전제가 옳다")
print(">>> horizon 에 따라 단조 증가하면 argmax 가 창 끝에 붙은 것 — 신호가 약하다는 뜻이다")
print(">>> max_corr_residual 이 장기 horizon 에서 커지면 그곳이 개선 여지다")

figure, axes = plt.subplots(1, 3, figsize=(16, 4.4))
axes[0].plot(lags, curve_target, "o-", label="vs target")
axes[0].plot(lags, curve_residual, "s-", label="vs residual (target - wind_19)")
axes[0].axvline(peak_lag_res, color="red", ls="--", label=f"residual peak {peak_lag_res:.0f}h")
for speed, style in [(350.0, ":"), (500.0, "-."), (700.0, ":")]:
    axes[0].axvline(AU_KM / speed / 3600, color="gray", ls=style, alpha=0.6)
axes[0].axvspan(0, MIN_LAG_HOURS, color="gray", alpha=0.15)
axes[0].set_xlabel("lag [hour]"); axes[0].set_ylabel("corr(CH area, y)")
axes[0].set_title("lag correlation (central meridian)")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

image = axes[1].imshow(table_residual, aspect="auto", cmap="coolwarm",
                       vmin=-np.nanmax(np.abs(table_residual)),
                       vmax=np.nanmax(np.abs(table_residual)),
                       extent=[-0.5, 19.5, 12.5, 0.5])
axes[1].set_xlabel("window index j (19 = T0)"); axes[1].set_ylabel("horizon index")
axes[1].set_title("corr with residual by (horizon, window)")
figure.colorbar(image, ax=axes[1])

axes[2].plot(HORIZONS, best_residual, "o-", label="residual")
axes[2].plot(HORIZONS, best_target, "s--", alpha=0.6, label="target")
axes[2].axhline(AU_KM / 500 / 3600, color="gray", ls="-.", label="tau(v=500)")
axes[2].set_xlabel("horizon [hour]"); axes[2].set_ylabel("best lag [hour]")
axes[2].set_title("optimal lag per horizon")
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(EDA_DIR / "ballistic_check.png", dpi=140); plt.show()

print("\nlag 별 표본 쌍 수 (양 끝은 쌍이 적어 신뢰도가 낮다)")
print(pd.DataFrame({"lag": lags, "pairs": pair_counts}).T.to_string(header=False))

## 6. 채널 · 임계 · 위도 밴드별 신호 세기

- Upendran et al. 2020: "**211 Å 가 더 나은 채널**" — 우리는 `193 AND 211` 만 쓴다
- Collin et al. 2025: 임계값이 지배 변수, 고위도는 기여 없음
- Upendran Grad-CAM: 고속풍 예측이 **먼 시점의 활성영역**에도 반응

각 조합의 lag 상관 최대값으로 신호 세기를 비교한다.

In [ ]:
records = []
for mode_index, mode in enumerate(MODES):
    for level in range(ch_area.shape[2]):
        series_ml = central_series(mode_index, level)
        if series_ml.std() < 1e-12:
            continue
        lags_ml, curve_ml, _, _ = lag_correlation(series_ml, block_residual)
        lag_ml, corr_ml = peak_of(lags_ml, curve_ml)
        records.append({"channel": mode, "level": LEVEL_NAMES[level],
                        "peak_corr": corr_ml, "peak_lag": lag_ml,
                        "area_mean": float(ch_area[:, mode_index, level].mean())})
channel_frame = pd.DataFrame(records).sort_values("peak_corr", key=np.abs, ascending=False)
print("=== 채널 × 임계 (중앙 자오선, 잔차 기준) ===")
print(channel_frame.round(3).to_string(index=False))

records = []
for lat_band in range(N_LAT):
    series_lat = central_series(AND_MODE, REFERENCE_LEVEL, lat_band=lat_band)
    if series_lat.std() < 1e-12:
        continue
    lags_lat, curve_lat, _, _ = lag_correlation(series_lat, block_residual)
    lag_lat, corr_lat = peak_of(lags_lat, curve_lat)
    records.append({"lat_band": lat_band,
                    "position": "north" if lat_band < N_LAT / 2 - 0.5 else
                                ("equator" if abs(lat_band - (N_LAT - 1) / 2) < 0.6 else "south"),
                    "peak_corr": corr_lat, "peak_lag": lag_lat,
                    "area_mean": float(ch_area[:, AND_MODE, REFERENCE_LEVEL,
                                               lat_band * N_LON + CENTRAL_LON].mean())})
latitude_frame = pd.DataFrame(records)
print("\n=== 위도 밴드별 (AND, dark<=0.45) ===")
print(latitude_frame.round(3).to_string(index=False))
print("\n>>> 적도 밴드가 압도적이면 Collin 의 '고위도 무기여' 가 우리 데이터에서도 성립")
print(">>> 남북 밴드의 상관이 비슷하면 적도 대칭 합(S_ij)으로 접어도 손실이 없다")

# 경도 열별 — 어느 경도가 언제 유효한지 (Collin: 과거로 갈수록 동쪽)
longitude_correlation = np.full((N_LON, len(lags)), np.nan)
for lon in range(N_LON):
    cells = [EQUATOR_LAT * N_LON + lon]
    area = ch_area[:, AND_MODE, REFERENCE_LEVEL][:, cells].mean(axis=1)[index_matrix]
    if area.std() < 1e-12:
        continue
    lags_lon, curve_lon, _, _ = lag_correlation(area, block_residual)
    longitude_correlation[lon, :len(curve_lon)] = curve_lon

figure, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for lon in range(N_LON):
    axes[0].plot(lags, longitude_correlation[lon], label=f"lon bin {lon}")
axes[0].set_xlabel("lag [hour]"); axes[0].set_ylabel("corr")
axes[0].set_title("lag correlation by longitude bin (east -> west)")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].bar(latitude_frame.lat_band, latitude_frame.peak_corr)
axes[1].set_xlabel("latitude band (0 = north)"); axes[1].set_ylabel("peak corr")
axes[1].set_title("signal strength by latitude band"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(EDA_DIR / "channel_latitude.png", dpi=140); plt.show()

## 7. 코로나홀 면적 시계열의 이상치

Collin et al. 2025 §2.1.1 —
> "remove those that exceed **five times the median absolute deviation of a sliding window of 10 observations**
> (Hampel filter) ... replace these excluded outliers by interpolating between the adjacent time steps."

우리 파이프라인에는 이상치 처리가 없다. 실제로 얼마나 튀는지 센다.

In [ ]:
def hampel_flags(series, window=10, sigma=5.0):
    n = len(series)
    flags = np.zeros(n, bool)
    half = window // 2
    for i in range(n):
        low, high = max(0, i - half), min(n, i + half + 1)
        chunk = series[low:high]
        median = np.median(chunk)
        deviation = 1.4826 * np.median(np.abs(chunk - median))
        if deviation > 1e-9 and abs(series[i] - median) > sigma * deviation:
            flags[i] = True
    return flags


total_area = ch_area[:, AND_MODE, REFERENCE_LEVEL].mean(axis=1)
flags = hampel_flags(total_area)
print(f"총 코로나홀 면적 이상치 {flags.sum()} / {len(flags)}  ({flags.mean():.2%})")
print(f"이상치 프레임이 최소 1개 포함된 20스텝 윈도우 비율: "
      f"{np.any(flags[index_matrix], axis=1).mean():.2%}")
print("\n>>> 윈도우 오염률이 5% 를 넘으면 Hampel 필터가 실효 있다")

figure, axis = plt.subplots(figsize=(13, 3.6))
axis.plot(total_area, lw=0.8, label="total CH area")
axis.plot(np.flatnonzero(flags), total_area[flags], "rx", ms=6, label="Hampel outlier")
axis.set_xlabel("unique image index"); axis.set_ylabel("CH area fraction")
axis.legend(); axis.grid(alpha=0.3); axis.set_title("CH area time series and outliers")
plt.tight_layout(); plt.savefig(EDA_DIR / "ch_outliers.png", dpi=140); plt.show()

## 8. 요약

In [ ]:
print("=" * 68)
print("P7 EDA 요약")
print("=" * 68)
print(f"1. train 연속 블록(50샘플 이상) : {len(TRAIN_BLOCKS)}개 "
      f"/ 전체 블록 {len(SAMPLE_BLOCKS['train'])}개")
print(f"   (11개 내외 · 각 7~8개월이면 월 기반 분할 = 1~8월 × 11년 확정)")
print(f"2. persistence 가 climatology 를 못 이기는 horizon : ", end="")
losing = val_baseline.horizon[val_baseline.persist_gain <= 0].tolist()
print(losing if losing else "없음 (전 구간에서 persistence 가 유효)")
print(f"3. 원반 반지름 변동 : {np.nanstd(radius)/np.nanmean(radius):.2%} "
      f"(진폭 {(np.nanmax(radius)-np.nanmin(radius))/np.nanmean(radius):.2%})")
print(f"4. 탄도 상관 최대 지연 (잔차 기준) : {peak_lag_res:.0f} h "
      f"-> 함의 속도 {implied_speed_res:.0f} km/s (상관 {peak_corr_res:.3f})")
print(f"   타깃 기준은 {peak_lag:.0f} h / {implied_speed:.0f} km/s (상관 {peak_corr:.3f})")
print(f"   horizon별 최적 지연 산포 : {np.nanstd(best_residual):.1f} h "
      f"(작을수록 탄도 정렬 전제가 옳음)")
long_horizon = horizon_frame[horizon_frame.horizon >= 36].max_corr_residual.mean()
short_horizon = horizon_frame[horizon_frame.horizon <= 18].max_corr_residual.mean()
print(f"   잔차 상관 평균 : 장기(36h+) {long_horizon:.3f} vs 단기(<=18h) {short_horizon:.3f}")
best_channel = channel_frame.iloc[0]
print(f"5. 최강 조합 : {best_channel.channel} / {best_channel.level} "
      f"(corr {best_channel.peak_corr:.3f} @ {best_channel.peak_lag:.0f}h)")
print(f"6. 위도 밴드 최강 : band {latitude_frame.loc[latitude_frame.peak_corr.abs().idxmax(), 'lat_band']}"
      f" / 적도 밴드 corr {latitude_frame.peak_corr.abs().max():.3f}")
print(f"7. 이상치 오염 윈도우 : {np.any(flags[index_matrix], axis=1).mean():.2%}")
print("=" * 68)
print("결과는 work/eda/ 아래 png 와 csv 로 저장됨")